# M11.6 — Pre-production temporal split audit

Before generating the G0/G1/R training datasets, we verify that the real-noise
environments assigned to different dataset splits do not reuse overlapping GPS
intervals.

This is a leakage check, not an additional noise-characterization study.

Each real-noise sample depends on both:

- the off-source PSD reference interval;
- the processing/noise interval.

For each unique environment we therefore define its temporal support as

$$
I =
[
\min(t_{\rm PSD,start}, t_{\rm proc,start}),
\max(t_{\rm PSD,end}, t_{\rm proc,end})
].
$$

Exact overlap between environments assigned to different splits would constitute
temporal data leakage and require revising the split assignment.

Temporal proximity without overlap is reported descriptively but is not treated
as leakage by itself.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
MANIFEST_PATH = Path(
    "/data/vserrano/cbc_pe_data/processed/"
    "m11_6_manifests/"
    "m11_6_master_experiment_manifest.csv"
)

df = pd.read_csv(
    MANIFEST_PATH
)

print("Manifest rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Manifest rows: 26000
Columns:
['source_index', 'mass_1', 'mass_2', 'spin_1z', 'spin_2z', 'inclination', 'ra', 'dec', 'polarization_angle', 'reference_distance_mpc', 'chirp_mass', 'total_mass', 'chi_eff', 'split', 'target_network_snr', 'source_id', 'file_group_id', 'block_id', 'noise_crop_id', 'center_gps', 'processing_start', 'processing_end', 'psd_start', 'psd_end', 'geocentric_time']


In [3]:
required_cols = [
    "split",
    "file_group_id",
    "block_id",
    "noise_crop_id",
    "psd_start",
    "psd_end",
    "processing_start",
    "processing_end",
]

missing = [
    c for c in required_cols
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing manifest columns: {missing}"
    )

In [4]:
df_env = (
    df[
        required_cols
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_env["support_start"] = np.minimum(
    df_env["psd_start"],
    df_env["processing_start"],
)

df_env["support_end"] = np.maximum(
    df_env["psd_end"],
    df_env["processing_end"],
)

print("Unique environments:", len(df_env))

display(
    df_env
    .groupby("split")
    .agg(
        n_file_groups=(
            "file_group_id",
            "nunique",
        ),
        n_blocks=(
            "block_id",
            "nunique",
        ),
        n_crops=(
            "noise_crop_id",
            "nunique",
        ),
    )
)

Unique environments: 4325


,n_file_groups,n_blocks,n_crops
split,,,
cal,3,39,568
test,3,25,363
train,20,200,2980
val,3,28,414


In [5]:
group_split_counts = (
    df_env
    .groupby("file_group_id")["split"]
    .nunique()
)

bad_groups = (
    group_split_counts[
        group_split_counts > 1
    ]
)

print(
    "file_group_id present in multiple splits:",
    len(bad_groups),
)

if len(bad_groups) > 0:
    display(
        bad_groups.to_frame(
            "n_splits"
        )
    )

file_group_id present in multiple splits: 0


In [6]:
env_records = (
    df_env[
        [
            "split",
            "file_group_id",
            "block_id",
            "noise_crop_id",
            "support_start",
            "support_end",
        ]
    ]
    .to_dict("records")
)

In [7]:
overlaps = []
nearest_pairs = []

for i in range(len(env_records)):
    a = env_records[i]

    for j in range(i + 1, len(env_records)):
        b = env_records[j]

        # Only cross-split comparisons matter.
        if a["split"] == b["split"]:
            continue

        # Same file_group across splits would already be bad,
        # but keep the temporal comparison general.
        overlap_s = (
            min(
                a["support_end"],
                b["support_end"],
            )
            -
            max(
                a["support_start"],
                b["support_start"],
            )
        )

        if overlap_s > 0:
            overlaps.append({
                "split_A":
                    a["split"],

                "split_B":
                    b["split"],

                "file_group_A":
                    a["file_group_id"],

                "file_group_B":
                    b["file_group_id"],

                "block_A":
                    a["block_id"],

                "block_B":
                    b["block_id"],

                "overlap_s":
                    float(overlap_s),
            })

            gap_s = 0.0

        else:
            gap_s = max(
                a["support_start"],
                b["support_start"],
            ) - min(
                a["support_end"],
                b["support_end"],
            )

        nearest_pairs.append({
            "split_A":
                a["split"],

            "split_B":
                b["split"],

            "file_group_A":
                a["file_group_id"],

            "file_group_B":
                b["file_group_id"],

            "block_A":
                a["block_id"],

            "block_B":
                b["block_id"],

            "gap_s":
                float(
                    max(gap_s, 0.0)
                ),
        })

In [8]:
df_overlap = pd.DataFrame(
    overlaps
)

print(
    "Cross-split temporal overlaps:",
    len(df_overlap),
)

if len(df_overlap) > 0:
    display(
        df_overlap
        .sort_values(
            "overlap_s",
            ascending=False,
        )
    )

Cross-split temporal overlaps: 0


In [9]:
df_nearest = (
    pd.DataFrame(
        nearest_pairs
    )
)

display(
    df_nearest
    .sort_values("gap_s")
    .head(20)
)

,split_A,split_B,file_group_A,file_group_B,block_A,block_B,gap_s
3788555,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788514,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788788,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788739,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788690,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788754,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788592,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788713,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788803,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875
3788851,train,test,16,17,M116_RB_00156,M116_RB_00157,33878.1875


In [10]:
display(
    df_nearest[
        "gap_s"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.50,
        ]
    )
)

count    4.599718e+06
mean     4.046194e+06
std      2.969618e+06
min      3.387819e+04
1%       7.040219e+04
5%       3.263752e+05
50%      3.508622e+06
max      1.190722e+07
Name: gap_s, dtype: float64

In [11]:
split_pair_summary = (
    df_nearest
    .assign(
        split_pair=lambda x:
            x.apply(
                lambda r:
                    " / ".join(
                        sorted([
                            str(r["split_A"]),
                            str(r["split_B"]),
                        ])
                    ),
                axis=1,
            )
    )
    .groupby(
        "split_pair"
    )["gap_s"]
    .agg(
        min_gap_s="min",
        median_gap_s="median",
    )
    .sort_values(
        "min_gap_s"
    )
)

display(
    split_pair_summary
)

,min_gap_s,median_gap_s
split_pair,,
test / train,33878.1875,3.629564e+06
cal / val,101610.1875,3.646022e+06
cal / test,198978.1875,3.810658e+06
cal / train,209048.1875,2.552939e+06
test / val,437942.1875,3.487458e+06
train / val,464874.1875,3.270591e+06


## Conclusion

The M11.6 real-noise split passes the temporal leakage audit.

- No `file_group_id` is shared across dataset splits.
- No GPS support interval overlaps between different splits.
- The PSD-reference and processing intervals used by train, validation,
  calibration and test are therefore temporally disjoint.

No additional temporal exclusion criterion is imposed, since proximity without
overlap is not treated as leakage in the present experiment.

The current split assignment is accepted for M11.6 dataset generation.